In [14]:
#!/usr/bin/env python3
"""
Label Studio Project Cloner
Clones a Label Studio project including configuration, tasks, annotations, and settings.
"""

import os
import json
from typing import Dict, List, Any, Optional
from label_studio_sdk import Client
from label_studio_sdk.core.api_error import ApiError
import traceback

class LabelStudioProjectCloner:
    def __init__(self, url: str, api_key: str):
        """
        Initialize the Label Studio client.
        
        Args:
            url: Label Studio server URL (e.g., 'http://localhost:8080')
            api_key: API key for authentication
        """
        self.client = Client(url=url, api_key=api_key)
        
    def get_project_details(self, project_id: int) -> Dict[str, Any]:
        """Get detailed information about the source project."""
        try:
            project = self.client.get_project(project_id)
            # Convert project object to dictionary
            project_dict = {}
            for attr in dir(project):
                if not attr.startswith('_') and not callable(getattr(project, attr)):
                    try:
                        project_dict[attr] = getattr(project, attr)
                    except:
                        continue
            return project_dict
        except ApiError as e:
            raise Exception(f"Failed to get project details: {e}")
    
    def get_project_config(self, project_id: int) -> str:
        """Get the labeling configuration of the source project."""
        try:
            project = self.client.get_project(project_id)
            # Try to get label_config attribute directly
            if hasattr(project, 'label_config'):
                return getattr(project, 'label_config', '')
            else:
                # Fallback: try to get it from the raw data
                return getattr(project, 'get_params', {}).get('label_config', '')
        except ApiError as e:
            raise Exception(f"Failed to get project configuration: {e}")
    
    def get_project_tasks(self, project_id: int) -> List[Dict[str, Any]]:
        """Get all tasks from the source project."""
        try:
            # Use the correct method to get tasks
            project = self.client.get_project(project_id)
            tasks = project.get_tasks()
            return tasks
        except ApiError as e:
            raise Exception(f"Failed to get project tasks: {e}")
        except AttributeError:
            # Fallback method if get_tasks() doesn't exist on project
            try:
                # Try using the client's make_request method directly
                response = self.client.make_request('GET', f'/api/projects/{project_id}/tasks/')
                return response
            except Exception as fallback_error:
                raise Exception(f"Failed to get project tasks: {fallback_error}")
    
    def get_project_annotations(self, project_id: int) -> List[Dict[str, Any]]:
        """Get all annotations from the source project."""
        try:
            annotations = []
            tasks = self.get_project_tasks(project_id)
            
            for task in tasks:
                task_id = task['id']
                try:
                    # Try to get annotations for this task
                    task_annotations = self.client.make_request('GET', f'/api/tasks/{task_id}/annotations/')
                    annotations.extend(task_annotations)
                except Exception as e:
                    print(f"Warning: Could not get annotations for task {task_id}: {e}")
                    continue
            
            return annotations
        except ApiError as e:
            raise Exception(f"Failed to get project annotations: {e}")
    
    def create_new_project(self, title: str, description: str = "", 
                          label_config: str = "", **kwargs) -> Dict[str, Any]:
        """Create a new Label Studio project."""
        try:
            project_data = {
                'title': title,
                'description': description,
                'label_config': label_config,
                **kwargs
            }
            
            new_project = self.client.create_project(**project_data)
            
            # Convert Project object to dictionary
            project_dict = {}
            for attr in dir(new_project):
                if not attr.startswith('_') and not callable(getattr(new_project, attr)):
                    try:
                        project_dict[attr] = getattr(new_project, attr)
                    except:
                        continue
            
            # Make sure we have the ID
            if hasattr(new_project, 'id'):
                project_dict['id'] = getattr(new_project, 'id')
            
            return project_dict
        except ApiError as e:
            raise Exception(f"Failed to create new project: {e}")
    
    def import_tasks_to_project(self, project_id: int, tasks: List[Dict[str, Any]]) -> None:
        """Import tasks to the new project."""
        try:
            # Clean tasks data - remove project-specific fields
            cleaned_tasks = []
            for task in tasks:
                cleaned_task = {k: v for k, v in task.items() 
                              if k not in ['id', 'project', 'created_at', 'updated_at', 
                                         'annotations', 'predictions', 'drafts']}
                cleaned_tasks.append(cleaned_task)
            
            if cleaned_tasks:
                project = self.client.get_project(project_id)
                project.import_tasks(cleaned_tasks)
                print(f"Imported {len(cleaned_tasks)} tasks")
        except ApiError as e:
            raise Exception(f"Failed to import tasks: {e}")
        except AttributeError:
            # Fallback method
            try:
                if cleaned_tasks:
                    response = self.client.make_request(
                        'POST', 
                        f'/api/projects/{project_id}/import',
                        json=cleaned_tasks
                    )
                    print(f"Imported {len(cleaned_tasks)} tasks")
            except Exception as fallback_error:
                raise Exception(f"Failed to import tasks: {fallback_error}")
    
    def import_annotations_to_project(self, project_id: int, 
                                    old_annotations: List[Dict[str, Any]],
                                    task_id_mapping: Dict[int, int]) -> None:
        """Import annotations to the new project with updated task IDs."""
        try:
            imported_count = 0
            for annotation in old_annotations:
                old_task_id = annotation.get('task')
                if old_task_id in task_id_mapping:
                    new_task_id = task_id_mapping[old_task_id]
                    
                    # Clean annotation data
                    cleaned_annotation = {
                        'task': new_task_id,
                        'result': annotation.get('result', []),
                        'was_cancelled': annotation.get('was_cancelled', False),
                        'ground_truth': annotation.get('ground_truth', False),
                        'lead_time': annotation.get('lead_time'),
                    }
                    
                    # Remove None values
                    cleaned_annotation = {k: v for k, v in cleaned_annotation.items() if v is not None}
                    
                    try:
                        # Use make_request to create annotation
                        response = self.client.make_request(
                            'POST', 
                            f'/api/tasks/{new_task_id}/annotations/',
                            json=cleaned_annotation
                        )
                        imported_count += 1
                    except Exception as e:
                        print(f"Warning: Failed to import annotation for task {new_task_id}: {e}")
                        continue
            
            print(f"Imported {imported_count} annotations")
        except Exception as e:
            raise Exception(f"Failed to import annotations: {e}")
    
    def get_task_id_mapping(self, old_tasks: List[Dict[str, Any]], 
                           new_project_id: int) -> Dict[int, int]:
        """Create a mapping between old and new task IDs."""
        try:
            new_tasks = self.get_project_tasks(new_project_id)
            
            # Create mapping based on task data content
            mapping = {}
            for i, old_task in enumerate(old_tasks):
                if i < len(new_tasks):
                    mapping[old_task['id']] = new_tasks[i]['id']
            
            return mapping
        except Exception as e:
            raise Exception(f"Failed to create task ID mapping: {e}")
    
    def clone_project(self, source_project_id: int, new_project_title: str,
                     new_project_description: str = "", 
                     include_annotations: bool = True) -> Dict[str, Any]:
        """
        Clone a complete Label Studio project.
        
        Args:
            source_project_id: ID of the source project to clone
            new_project_title: Title for the new project
            new_project_description: Description for the new project
            include_annotations: Whether to copy annotations
            
        Returns:
            Dictionary containing information about the cloned project
        """
        print(f"Starting to clone project {source_project_id}...")
        
        # Get source project details
        print("Fetching source project details...")
        source_project = self.get_project_details(source_project_id)
        
        # Get project configuration
        print("Fetching project configuration...")
        label_config = self.get_project_config(source_project_id)
        
        # Get tasks
        print("Fetching project tasks...")
        tasks = self.get_project_tasks(source_project_id)
        
        # Get annotations if requested
        annotations = []
        if include_annotations:
            print("Fetching project annotations...")
            annotations = self.get_project_annotations(source_project_id)
        
        # Create new project
        print("Creating new project...")
        project_params = {
            'title': new_project_title,
            'description': new_project_description or source_project.get('description', ''),
            'label_config': label_config,
        }
        
        # Add optional parameters if they exist in source project
        optional_params = [
            'color', 'maximum_annotations', 'sampling', 'show_instruction', 
            'show_skip_button', 'enable_empty_annotation', 'show_annotation_history'
        ]
        
        for param in optional_params:
            if param in source_project:
                project_params[param] = source_project[param]
        
        # Set some safe defaults
        project_params.update({
            'is_published': False,  # Start as unpublished
            'maximum_annotations': source_project.get('maximum_annotations', 1),
        })
        
        # Remove None values
        project_params = {k: v for k, v in project_params.items() if v is not None}
        
        new_project = self.create_new_project(**project_params)
        
        # Get project ID safely
        if isinstance(new_project, dict):
            new_project_id = new_project['id']
        else:
            # If it's still a Project object, get the ID attribute
            new_project_id = getattr(new_project, 'id')
        
        print(f"Created new project with ID: {new_project_id}")
        
        # Import tasks
        if tasks:
            print("Importing tasks...")
            self.import_tasks_to_project(new_project_id, tasks)
            
            # Import annotations if requested
            if include_annotations and annotations:
                print("Creating task ID mapping...")
                task_id_mapping = self.get_task_id_mapping(tasks, new_project_id)
                
                print("Importing annotations...")
                self.import_annotations_to_project(new_project_id, annotations, task_id_mapping)
        
        result = {
            'source_project_id': source_project_id,
            'new_project_id': new_project_id,
            'new_project_title': new_project_title,
            'tasks_count': len(tasks),
            'annotations_count': len(annotations) if include_annotations else 0,
            'success': True
        }
        
        print("Project cloning completed successfully!")
        print(f"Source project: {source_project_id}")
        print(f"New project ID: {new_project_id}")
        print(f"Tasks imported: {len(tasks)}")
        if include_annotations:
            print(f"Annotations imported: {len(annotations)}")
        
        return result


def main():
    # TARGET INSTANCE CONFIG
    TARGET_LS_URL = 'http://129.97.250.147:8080/'
    TARGET_API_TOKEN = 'ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d'
    # Configuration - Update these values
    LABEL_STUDIO_URL = "http://129.97.250.147:8080/"  # Your Label Studio server URL
    API_KEY = "ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d"  # Your Label Studio API key
    SOURCE_PROJECT_ID = 113  # ID of the project you want to clone
    NEW_PROJECT_TITLE = "Cloned Project"  # Title for the new project
    NEW_PROJECT_DESCRIPTION = "This is a cloned project"  # Description for the new project
    INCLUDE_ANNOTATIONS = True  # Set to False if you don't want to copy annotations
    
    try:
        cloner = LabelStudioProjectCloner(LABEL_STUDIO_URL, API_KEY)
        result = cloner.clone_project(
            source_project_id=SOURCE_PROJECT_ID,
            new_project_title=NEW_PROJECT_TITLE,
            new_project_description=NEW_PROJECT_DESCRIPTION,
            include_annotations=INCLUDE_ANNOTATIONS
        )
        
        print("\nCloning Summary:")
        print(json.dumps(result, indent=2))
        
    except Exception as e:
        print(f"Error: {e}")
        print(traceback.format_exc())
        exit(1)


if __name__ == "__main__":
    main()

Starting to clone project 113...
Fetching source project details...
Fetching project configuration...
Fetching project tasks...
Fetching project annotations...
Creating new project...
Created new project with ID: 118
Importing tasks...
Imported 9 tasks
Creating task ID mapping...
Importing annotations...
Error: Failed to import annotations: 'bytes' object has no attribute 'get'
Traceback (most recent call last):
  File "/tmp/ipykernel_2020576/2048318230.py", line 159, in import_annotations_to_project
    old_task_id = annotation.get('task')
AttributeError: 'bytes' object has no attribute 'get'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_2020576/2048318230.py", line 326, in main
    result = cloner.clone_project(
  File "/tmp/ipykernel_2020576/2048318230.py", line 291, in clone_project
    self.import_annotations_to_project(new_project_id, annotations, task_id_mapping)
  File "/tmp/ipykernel_2020576/2048